# Archived docking workflow — curated copy

Source: private original Colab source (identifier withheld)

Execution outputs and execution state were stripped. The destructive `rm -rf` cell was disabled; all other code is preserved for static review. **Do not run without checking paths, dependencies, filenames, Vina arguments and input preparation.** An original notebook is not evidence that its docking scores were independently validated.


# Remove previous folder (USE WITH CAUTION)
* replace `<FOLDER>` in the next cell with the name of the folder on your drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# DISABLED BY CURATION (DESTRUCTIVE): !rm -rf  /content/drive/MyDrive/trial

# Download, install programs, and connect to drive (run without modifications)

* vina: https://github.com/ccsb-scripps/AutoDock-Vina
* rdkit: https://github.com/rdkit/rdkit
* meeko: https://github.com/forlilab/Meeko
* scrubber: https://github.com/forlilab/molscrub
* mdanalysis: https://www.mdanalysis.org/
* xlsxwriter: https://xlsxwriter.readthedocs.io/
* nglview: https://github.com/nglviewer/nglview
* CDPKit: https://cdpkit.org/introduction.html
* prody: https://github.com/prody/ProDy


In [ ]:
%%bash

# vina
wget https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.7/vina_1.2.7_linux_x86_64
chmod +x /content/vina_1.2.7_linux_x86_64

# rdkit
pip install rdkit

# meeko
git clone https://github.com/forlilab/Meeko.git
cd Meeko
git checkout develop
pip install .
pip install gemmi
cd /content

# scrubber
git clone https://github.com/forlilab/molscrub.git
cd /content/molscrub
pip install -e .
cd /content

# mdanalysis
pip install --upgrade MDAnalysis

# xlsxwriter
pip install xlsxwriter

# nglview
pip install nglview==3.0.8

# CDPKit
wget https://github.com/molinfo-vienna/CDPKit/releases/download/v1.2.3/CDPKit-1.2.3-Linux-x86_64.sh
chmod +x ./*
echo y | /content/CDPKit-1.2.3-Linux-x86_64.sh

# prody
pip install prody

# openbabel
sudo apt update
sudo apt install openbabel

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

# Prepare receptor

In [ ]:
#@title Make a folder for docking files (it will have all files related to docking)

#@markdown * Change `<FOLDER_NAME>` to the name of the folder you want
#@markdown * **DO NOT ADD ANY SYMBOL OR SPACE IN THE NAME OF THE FOLDER**
#@markdown * Then Run the cell
import os
BASE = '/content/drive/MyDrive/zzz' #@param {type:"string"}
os.system(f'mkdir {BASE}')

In [ ]:
#@title Determine the center of the active site (you need to upload the .pdb protein file in the BASE folder)

#@markdown * a trial pdb file: https://www.rcsb.org/structure/1IEP <br>
#@markdown * 1- Download the .pdb <br> 2- Remove any unwanted molecules (water, ions, etc..) using PyMOL <br> 3- Separate the protein and the ligand in two different .pdb files using PyMOL <br> 4- Save the cleaned complex in another .pdb file <br> 5- Upload the receptor and ligand files in the `BASE` directory <br>
#@markdown * In pymol, determine the residues numbers around the ligand within 4 Ang <br>and write these numbers in **active_site_aa** variable. <br>
#@markdown * They should be in this format: 1,2,3,4,5,...
import MDAnalysis as mda

#@markdown write the receptor file name (after uploading it) without the .pdb <br>
rec_file_name = 'prot' #@param {type:"string"}
uni = mda.Universe(BASE+f'/{rec_file_name}.pdb')

#@markdown This active site is for **1IEP** <br>
#@markdown If you downloaded another pdb, you need to change the active site amino acids<br>

active_site_aa = '248,253,256,269,271,286,289,290,299,313,315,317,318,321,360,361,362,370,380,381,382' #@param {type:"string"}
active_resn = active_site_aa.replace(',', ' or resid ')
active_site = uni.select_atoms('resid '+active_resn)

act_site_center = active_site.center(weights=None)
print(f'The center of the active site is: {round(act_site_center[0],3)} {round(act_site_center[1],3)} {round(act_site_center[2],3)}')

In [ ]:
#@title prepare receptor
import os, subprocess

rec = BASE+f'/{rec_file_name}.pdb'
out = BASE+f'/{rec_file_name}_prepared'

#@markdown Enter the box size in angstrom (you can select it as you want)<br>
#@markdown It should be in this format: X Y Z (each two numbers are separated by a space)
box_size = '30 30 30' #@param {type:"string"}

#@markdown Enter the box center from the previous cell output<br>
#@markdown It should be in this format: X Y Z (each two numbers are separated by a space)
box_center = '0.238 -11.634 -15.294' #@param {type:"string"}

box_size_ls = box_size.split()
box_center_ls = box_center.split()
# change the box center from the previous cell values
# you can change the box size as you want. not less than 20 20 20

# -i: the name of the input file WITH the extension
# -o: the name of the output file WITHOUT the extension
# --default_altloc A: select the A conformation for amino acids with more than one
# -p: prepare pdbqt and output a file with the name after -o and ending with .pdbqt
# -v: write a file for vina containing the box dimensions and center
# --box_size X Y Z: size of grid box (x, y, z) in Angstrom
# --box_center X Y Z: center of grid box (x, y, z) in Angstrom
# -f A:315,316: make these aa as flexible

command = ['mk_prepare_receptor.py',
            '-i', f'{rec}',
            '-o', f'{out}',
            '--default_altloc', 'A',
            '-p', '-v',
            '--box_size', f'{int(box_size_ls[0])}', f'{int(box_size_ls[1])}', f'{int(box_size_ls[2])}',
            '--box_center', f'{float(box_center_ls[0])}', f'{float(box_center_ls[1])}', f'{float(box_center_ls[2])}'
            ]
res = subprocess.run(command, capture_output=True, text=True, check=True)
print(res.stderr)
print(res.stdout)

# !mk_prepare_receptor.py -i /content/drive/MyDrive/docking_trial/rec.pdb \
# -o /content/drive/MyDrive/docking_trial/rec_prepared --default_altloc A -p -v \
# --box_size 30 30 30 \
# --box_center 15.567 54.349 15.316

# os.system(f'mk_prepare_receptor.py -i {rec} \
# -o {out} --default_altloc A -p -v \
# --box_size {int(box_size_ls[0])} {int(box_size_ls[1])} {int(box_size_ls[2])} \
# --box_center {float(box_center_ls[0])} {float(box_center_ls[1])} {float(box_center_ls[2])}')

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()
import nglview as ngv
from nglview import NGLWidget

#@title Visualizing the box
#@markdown Enter the name of the ligand file without the .pdb

lig = 'ligand1' #@param {type:'string'}

view = NGLWidget()
view.add_structure(ngv.FileStructure(f'{BASE}/{rec_file_name}.pdb'))
view.add_structure(ngv.FileStructure(f'{BASE}/{lig}.pdb'))
view.add_structure(ngv.FileStructure(f'{BASE}/{rec_file_name}_prepared.box.pdb'))
view.add_line('not protein')
view

# prepare ligand

In [ ]:
#@title Prepare folders (run without modifications)
#@markdown This will generate three folders (`2D`, `3D`, and `pdbqts`) in the `BASE` folder
#@markdown * `2D` folder will contain the initial structure (SMILES or 2D sdf)
#@markdown * `3D` folder will contain the conformations of the ligands from CDPKit (3D sdf)
#@markdown * `pdbqts` folder will contain the prepared ligands as pdbqt files

two_D = f'{BASE}/2D'
three_D = f'{BASE}/3D'
pdbqts = f'{BASE}/pdbqts'

# make 2D folder for the original 2D mols
os.system(f'mkdir {two_D}')

# make 3D folder for the generated 3D mols from scrub.py
os.system(f'mkdir {three_D}')

# make pdbqts folder for the prepared 3D mols
os.system(f'mkdir {pdbqts}')

In [ ]:
#@title Generation of different conformation (using CDPKit)

#@markdown * **THE LIGANDS FILE NAME MUST NOT CONTAIN STRANGE SYMBOLS/SPACES. YOU CAN USE ONLY THE underscore (_)**
#@markdown * **Upload the ligand files in the 2D folder as `.sdf`/`.smi`/`.smiles`/`.mol`/`.mol2` files. Then run this cell**

#@markdown Parameters you can change and their meaning:
#@markdown * **--e-window** (`E_window`): Energy window for generated conformers. The conformation with the highest energy will have a difference of this value compared to the global minima. Default: 15.0
#@markdown * **--rmsd** (`RMSD`): Minimum RMSD for output conformer selection. The higher the more diverse. Default: 0.5
#@markdown * **--max-num-out-confs** (`Max_Confs`): Maximum number of output conformers per molecule. Default: 100
#@markdown * **--from-scratch**: whether to discard the initial 3D configuration. Default: true
#@markdown * **--timeout** (`Timeout`): maximum time of generation for each molecule. Default: 3600 sec
#@markdown * **--conf-idx-suffix**: Append conformer index to the title of multiconf. Default: false
#@markdown * **--num-threads** (`Threads`): number of cores to use


import os, subprocess

E_window = '20' #@param {type:"string"}
RMSD = '1.5' #@param {type:"string"}
Max_Confs = '600' #@param {type:"string"}
Timeout =  '60' #@param {type:"string"}
Threads = '2' #@param {type:"string"}


files = [i for i in os.listdir(two_D) if '.ipynb' not in i]
for i in files:
  print(f'Working on {i}')
  command = ['/opt/CDPKit/Bin/confgen',
             '-i', f'{two_D}/{i}',
             '-o', f'{three_D}/{i.split(".")[0]}.sdf',
             '--e-window', E_window,
             '--rmsd', RMSD,
             '--max-num-out-confs', Max_Confs,
             '--from-scratch', 'true',
             '--timeout', Timeout,
             '--conf-idx-suffix', 'true',
             '--num-threads', Threads
             ]
  res = subprocess.run(command, capture_output=True, text=True, check=True)
  print(res.stderr)
  print(res.stdout)
#   os.system(f"/opt/CDPKit/Bin/confgen -i {two_D}/{i} -o {three_D}/{i.split('.')[0]}.sdf \
# --e-window 20 --rmsd 2.0 \
# --max-num-out-confs 100  \
# --from-scratch true --timeout 60 \
# --conf-idx-suffix true \
# --num-threads 2")


#@markdown * After using CDPKit (running this cell), go to the "to PDBQT" section

# 📦 SDF Splitting Pipeline (RDKit)

## 🔹 Define input and output folders
## 🔹 Create output directory
## 🔹 Loop through SDF files
## 🔹 Split each conformer into a separate SDF
## 🔹 Preserve full metadata (V3000)
## 🔹 Count total generated files


In [ ]:
#@title codes
import os
from rdkit import Chem

input_dir = three_D
split_dir = os.path.join(three_D, "SPLIT_fixed")

os.makedirs(split_dir, exist_ok=True)

total = 0

print("Splitting with full metadata...")

for sdf in os.listdir(input_dir):
    if not sdf.endswith(".sdf"):
        continue

    suppl = Chem.SDMolSupplier(os.path.join(input_dir, sdf), sanitize=False, removeHs=False)
    base = sdf.replace(".sdf","")

    i = 1
    for mol in suppl:
        if mol is None:
            continue

        outpath = os.path.join(split_dir, f"{base}_conf{i}.sdf")
        w = Chem.SDWriter(outpath)
        w.SetForceV3000(True)   # مهم جداً
        w.write(mol)
        w.close()

        i += 1
        total += 1

print("Done splitting.")
print("Total:", total)
print("Folder:", split_dir)


# 🧬 Ligand Preparation Pipeline (SDF → PDBQT)

## 🔹 Input Conformers Directory
- Folder containing SDF conformer files generated earlier.

## 🔹 Output Directory for PDBQT
- Folder where converted PDBQT files will be saved.

## 🔹 Loop Through Conformers
- Read each .sdf file from the input directory.

## 🔹 Run mk_prepare_ligand.py
- Convert each SDF file into one or multiple PDBQT files.
- Preserve atom mapping with `--add_index_map`.
- Use `--multimol_prefix` to keep naming structured.

## 🔹 Collect Total Output Files
- Count how many PDBQT files were generated.


In [ ]:
#@title codess
import os, subprocess

input_dir = split_dir   # فولدر الكونفورمرز
output_dir = pdbqts

os.makedirs(output_dir, exist_ok=True)

files = [os.path.join(input_dir, f) for f in os.listdir(input_dir) if f.endswith(".sdf")]

print(f"Found {len(files)} conformers")

print("Converting SDF → PDBQT ...")

for f in files:
    base = os.path.splitext(os.path.basename(f))[0]
    print("Processing:", base)

    command = [
        "mk_prepare_ligand.py",
        "-i", f,
        "--multimol_outdir", output_dir,
        "--multimol_prefix", base,  # ← مهم جداً لتسمية كل ملف
        "--add_index_map"
    ]

    res = subprocess.run(command, text=True, capture_output=True)
    if res.stderr.strip():
        print("ERROR:", res.stderr)
    else:
        print("OK")

print("\nDone.")
print("Total PDBQT files:", len([x for x in os.listdir(output_dir) if x.endswith('.pdbqt')]))


# perform docking

* If the docking stopped (runtime disconnected/internet issus/etc..), you can start DIRECTLY from this cell. <br>It will continue docking from the last compound
* **YOU MUST BE CONNECTED TO DRIVE TO CONTINUE**

#in split fixed added hydrogen bo

In [ ]:
#@title Docking

#@markdown * You can change the value of `EXHAUSTIVENESS`

import os, time, subprocess

def split_pdbqt(file):
  with open(file) as f:
    models = f.read().split('ENDMDL')
  return models[:-1]
def process_pdbqt_file(model):
    """
    Processes a PDB file to reorder lines based on an index map.

    Args:
        file_path (str): The path to the PDB file.

    Returns:
        list: A list of reordered lines, or an empty list if the file
              couldn't be processed.
    """
    atom_dict = {}
    index_map_lines = []
    H_atoms = []
    max_numb = 0
    # Step 1-4: Read the file and populate the dictionary and list
    lines = model.splitlines()
    for line in lines:
        line_stripped = line.strip()
        # Step 1-3
        if line_stripped.startswith("ATOM"):
            parts = line_stripped.split()
            if 'H' in parts[2]:
              H_atoms.append(line_stripped)
              continue
            if len(parts) > 1 and parts[1].isdigit():
                atom_dict[int(parts[1])] = line_stripped[:-10]
        # Step 4
        elif line_stripped.startswith("REMARK INDEX MAP"):
            index_map_lines.append(line_stripped)
    # print(atom_dict)
    # print(index_map_lines)
    # If no ATOM lines are found, we can't proceed.
    if not atom_dict:
        return []

    # Step 5: Create the reordered list with empty strings
    reordered_lines = [''] * (len(atom_dict)+len(H_atoms))

    # Step 6: Process the index map and populate the reordered list
    for index_map_line in index_map_lines:
        # Split the line to get only the numbers after "INDEX MAP"
        parts = index_map_line.split()
        # The numbers start from the 3rd element (index 2)
        if len(parts) > 2:
            numbers = parts[3:]
            # Process pairs of numbers
            for i in range(0, len(numbers), 2):
                if i + 1 < len(numbers):
                    first_num = int(numbers[i])
                    second_num = int(numbers[i+1])

                    # Get the line from the dictionary using the second number as key
                    line_to_add = atom_dict.get(second_num)

                    # Use the first number minus 1 as the index for the new list
                    if line_to_add and 0 <= first_num - 1 < len(reordered_lines):
                        reordered_lines[first_num - 1] = line_to_add+'\n'#.replace(str(second_num), str(first_num),1)
    for i in reordered_lines:
      if len(i) < 2:
        idx2 = 0
        for idx1, elem in enumerate(reordered_lines):
          if elem == '':
            second_num = H_atoms[idx2].split()[1]
            reordered_lines[idx1] = H_atoms[idx2][:-10]+'\n'#.replace(str(second_num), str(idx1+1),1)
            idx2+=1
      break
    return reordered_lines

def renumber_lines(lines):
  renumbered_lines = []
  line_number = 1

  # Iterate through each line and renumber it
  for line in lines:
      # Ensure the line starts with ATOM
      if line.startswith("ATOM"):
          # The atom serial number is in columns 7-11 (indices 6-10)
          # Format the new number to be right-justified in 5 characters
          new_number_str = f"{line_number:>5}"

          # Replace the old number with the new one
          new_line = line[:6] + new_number_str + line[11:]
          renumbered_lines.append(new_line)
          line_number += 1
      else:
          # If the line doesn't start with ATOM, keep it as is
          renumbered_lines.append(line)
  renumbered_lines = ''.join(renumbered_lines)
  return renumbered_lines

def main(pdbqt_file):
  models = split_pdbqt(f'{DIR_outs}/{pdbqt_file}_out.pdbqt')
  for idx, model in enumerate(models):
    lines = process_pdbqt_file(model)
    lines = renumber_lines(lines)
    with open(f'{DIR_outs}/{pdbqt_file}_out_{idx+1}.pdb', 'w') as f:
      f.writelines(lines)



EXHAUSTIVENESS = '4' #@param {type:"string"}

try:
  if BASE == '':
    pass
except:
    print('YOU MUST RUN THE FIRST CELLS THAT DOWNLOAD THE PROGRAMS AND THAT DEFINES THE BASE FOLDER (inside the PREPARE RECEPTOR SECTION) \n THEN RUN THIS CELL AGAIN\n\n')

box = [i for i in os.listdir(BASE) if i.endswith('.box.txt') ]
rec = [i for i in os.listdir(BASE) if  i.endswith('prepared.pdbqt') ]

two_D = f'{BASE}/2D'
three_D = f'{BASE}/3D'
DIR_ligs = f'{BASE}/pdbqts'
DIR_outs = f'{BASE}/docked'
DIR_finished = f'{BASE}/finished'
os.system(f'mkdir {DIR_outs}')
os.system(f'mkdir {DIR_finished}')

pdbqts =[i for i in os.listdir(DIR_ligs) if i.endswith('pdbqt') if '.ipynb' not in i]

for pdbqt in pdbqts:
  start = time.time()
  print(f'working on {pdbqt}')
  # add --flex {DIR_rec}/rec_prepared_flex.pdbqt \ if you are performing flexible docking
  command = f'/content/vina_1.2.7_linux_x86_64 \
  --receptor {BASE}/{rec[0]} \
  --ligand {DIR_ligs}/{pdbqt} \
  --config {BASE}/{box[0]} \
  --exhaustiveness={EXHAUSTIVENESS} \
  --out {DIR_outs}/{pdbqt[:-6]}_out.pdbqt > {DIR_outs}/{pdbqt[:-6]}_log.log'

  res = subprocess.run(command, capture_output=True, text= True, shell= True)
  print('outs:', res.stdout)
  print('errors:', res.stderr)
  main(pdbqt[:-6])
  # os.system(f'mk_export.py {DIR_outs}/{pdbqt[:-6]}_out.pdbqt -s {DIR_outs}/{pdbqt[:-6]}_out.sdf')
  os.system(f'mv {DIR_ligs}/{pdbqt} {DIR_finished}')
  end = time.time()
  print(f'it took {round((end - start)/60, 2)} minutes')


# Compress the docked folder
print('Now compressing ALL the docking folders and important files')
os.system(f'tar -zcf {BASE}/docked_poses.tgz {two_D} {three_D} {DIR_ligs} {DIR_outs} {DIR_finished} {BASE}/*.pdbqt {BASE}/*.txt {BASE}/.pdb')

print('''\n\nTo calculate the redocking RMSD using VMD follow these steps:
1- Download the .tgz file containing all the data an extract it
2- Go to the docked folder. you will find separated .pdb files for each pose
3- Open the pose you want in VMD
4- Open the cocrystalized ligand (separated from the original pdb file) and save it as pdb in pymol
5- Open the pdb file from step 4 in VMD
6- Calculate the RMSD between the two molecules in VMD''')

# parsing log files

* This will generate an .xlsx file in the BASE folder
* You can change the value of the xlsx_name

In [ ]:
#@title parsing log files
import os, xlsxwriter

xlsx_name = 'comps2zzz' #@param {type:'string'}

# getting the names of the log files
logs = [DIR_outs+'/'+f for f in os.listdir(DIR_outs) if f.endswith('.log')]

print(f'you have: {len(logs)} logs')

drug_affinity = {}
errors = []

for log in logs:
    with open(log) as f:
        data = f.read()

    idx_start = data.find('   1    ')
    idx_end = len(data)

    affinities = []
    res = data[idx_start:idx_end]
    modes = res.splitlines()[:-1]

    for mode in modes:
        try:
            affinity = mode.split()[1]
        except:
            print('error with', log)
            errors.append(log)
            continue

        # التأكد أن القيمة رقمية
        try:
            float(affinity)
        except:
            print("قيمة affinity غير رقمية في:", log)
            continue

        affinities.append(affinity)

    drug_affinity[log.split('/')[-1][:-4]] = affinities


# ---------------------------
#     SORT SAFE VERSION
# ---------------------------

def safe_key(x):
    """إرجاع قيمة الفرز الأولى أو inf إذا ما فيه قيم."""
    if len(x[1]) > 0:
        return float(x[1][0])
    return float("inf")   # يجعل العناصر الفارغة في آخر القائمة


list_sorted = list(drug_affinity.items())
list_sorted.sort(key=safe_key)


# ---------------------------
#     كتابة النتائج في Excel
# ---------------------------

workbook = xlsxwriter.Workbook(f'{BASE}/{xlsx_name}.xlsx')
worksheet = workbook.add_worksheet()

for row, (name, affs) in enumerate(list_sorted):
    worksheet.write(row, 0, name)
    for col, aff in enumerate(affs):
        worksheet.write(row, col + 1, float(aff))

workbook.close()

print("تم إنشاء ملف Excel بنجاح!")


In [ ]:
for name, affs in drug_affinity.items():
    if len(affs) == 0:
        print("⚠ لا توجد نتائج في ملف:", name)
